# Ejemplo sencillo de detección de objetivos con OpenCV en manufactura

## Objetivo del ejercicio

Simular una estación de inspección visual en una línea de manufactura. Cada imagen representa una banda transportadora con piezas circulares. El programa deberá:

1. Detectar la ubicación de cada pieza.
2. Dibujar un rectángulo alrededor de cada objetivo.
3. Clasificar la pieza como **conforme** o **defectuosa**.
4. Comparar el resultado automático contra las etiquetas conocidas del dataset.

Este es un ejercicio didáctico. En una planta real, la cámara, la iluminación, la calibración y las tolerancias deben diseñarse específicamente para el proceso.

## Contexto del proceso

Imaginemos una cámara colocada sobre una banda transportadora. Las piezas buenas tienen color verde y las piezas con defecto visual tienen color rojo. El fondo de la banda es gris. Esta diferencia de color permite construir una solución transparente y fácil de entender: OpenCV separará los colores del fondo, encontrará los contornos y usará el color promedio de cada contorno para decidir la clase.

### Supuestos clave

- El fondo es relativamente uniforme.
- Las piezas no se tocan entre sí.
- La cámara permanece fija.
- El color es una señal válida del defecto en este ejemplo.
- El dataset es sintético y se genera con una semilla fija; por eso puede reproducirse exactamente en Colab.

## 1. Preparar el entorno

Colab normalmente ya incluye estas bibliotecas. La primera línea instala o actualiza OpenCV, NumPy, pandas y matplotlib si hiciera falta. `numpy` manejará imágenes como matrices, `pandas` organizará las etiquetas y `matplotlib` mostrará los resultados. La variable `RANDOM_SEED` hace que el dataset sea reproducible.

In [ ]:
!pip -q install opencv-python-headless pandas matplotlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
IMAGE_WIDTH, IMAGE_HEIGHT = 640, 360
DATASET_SIZE = 40
MIN_CONTOUR_AREA = 250
print('OpenCV:', cv2.__version__)

## 2. Crear el dataset sintético dentro del notebook

Esta celda genera 40 imágenes y sus etiquetas. Cada imagen contiene entre 3 y 8 piezas circulares. La etiqueta de cada pieza guarda su clase y su caja real (`x`, `y`, `width`, `height`). Un pequeño ruido visual hace que el ejemplo se parezca más a una captura real y evita que la solución dependa únicamente de imágenes perfectas.

La carpeta `/content/dataset_opencv_manufactura` es propia de Colab. No se descarga ningún dataset externo: el código que genera las imágenes forma parte del dataset y permite reconstruirlo desde cero.

In [ ]:
dataset_dir = Path('/content/dataset_opencv_manufactura')
dataset_dir.mkdir(parents=True, exist_ok=True)
annotations = []

for image_id in range(DATASET_SIZE):
    image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH, 3), (125, 125, 125), dtype=np.uint8)
    # Variación ligera del fondo para simular iluminación no perfectamente uniforme.
    background_noise = rng.normal(0, 3, image.shape).astype(np.int16)
    image = np.clip(image.astype(np.int16) + background_noise, 0, 255).astype(np.uint8)
    piece_count = int(rng.integers(3, 9))
    centers = []

    for piece_id in range(piece_count):
        radius = int(rng.integers(22, 36))
        # Rechazamos posiciones demasiado cercanas para evitar piezas pegadas.
        for _ in range(100):
            center_x = int(rng.integers(radius + 10, IMAGE_WIDTH - radius - 10))
            center_y = int(rng.integers(radius + 10, IMAGE_HEIGHT - radius - 10))
            if all(np.hypot(center_x - old_x, center_y - old_y) > 2.4 * radius for old_x, old_y, _ in centers):
                break
        is_defective = bool(rng.random() < 0.25)
        class_name = 'defectuosa' if is_defective else 'conforme'
        color_bgr = (40, 60, 220) if is_defective else (50, 190, 70)
        cv2.circle(image, (center_x, center_y), radius, color_bgr, -1)
        # Brillo pequeño para que la pieza tenga apariencia tridimensional.
        cv2.circle(image, (center_x - radius // 3, center_y - radius // 3), max(3, radius // 6), (220, 220, 220), -1)
        annotations.append({'image_id': image_id, 'piece_id': piece_id, 'class_name': class_name, 'x': center_x - radius, 'y': center_y - radius, 'width': 2 * radius, 'height': 2 * radius})
        centers.append((center_x, center_y, radius))

    image_path = dataset_dir / f'pieza_{image_id:03d}.png'
    cv2.imwrite(str(image_path), image)

ground_truth = pd.DataFrame(annotations)
ground_truth.to_csv(dataset_dir / 'annotations.csv', index=False)
print(f'Dataset creado: {DATASET_SIZE} imágenes y {len(ground_truth)} piezas etiquetadas.')
display(ground_truth.head())

## 3. Revisar visualmente el dataset

Antes de automatizar, conviene mirar algunos ejemplos. La función `cv2.imread` lee una imagen en formato BGR; como matplotlib espera RGB, se usa `cv2.cvtColor` para corregir el orden de canales. Si esta conversión se omite, los colores pueden verse intercambiados aunque el algoritmo de OpenCV siga trabajando.

In [ ]:
sample_paths = sorted(dataset_dir.glob('pieza_*.png'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for axis, image_path in zip(axes.ravel(), sample_paths):
    image_bgr = cv2.imread(str(image_path))
    axis.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axis.set_title(image_path.name)
    axis.axis('off')
plt.suptitle('Muestras del dataset sintético', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Segmentar las piezas por color

La segmentación convierte la imagen en una máscara binaria: píxeles blancos representan posibles piezas y píxeles negros representan fondo. Trabajamos en HSV porque separa mejor el tono (`H`), la intensidad del color (`S`) y la luminosidad (`V`) que el espacio RGB.

Se crean dos máscaras: una para verde y otra para rojo. En HSV el rojo aparece cerca de los extremos del rango de tono, por eso se unen dos intervalos rojos. `cv2.morphologyEx` limpia pequeños puntos de ruido; `MORPH_OPEN` elimina puntos aislados y `MORPH_CLOSE` rellena pequeños huecos.

In [ ]:
def build_piece_mask(image_bgr):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    green_lower = np.array([35, 60, 40])
    green_upper = np.array([90, 255, 255])
    red_lower_1 = np.array([0, 70, 40])
    red_upper_1 = np.array([12, 255, 255])
    red_lower_2 = np.array([165, 70, 40])
    red_upper_2 = np.array([179, 255, 255])
    green_mask = cv2.inRange(hsv, green_lower, green_upper)
    red_mask = cv2.inRange(hsv, red_lower_1, red_upper_1) | cv2.inRange(hsv, red_lower_2, red_upper_2)
    combined_mask = green_mask | red_mask
    kernel = np.ones((5, 5), np.uint8)
    return cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, kernel)

example_image = cv2.imread(str(sample_paths[0]))
example_mask = build_piece_mask(example_image)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(example_image, cv2.COLOR_BGR2RGB)); axes[0].set_title('Imagen original'); axes[0].axis('off')
axes[1].imshow(example_mask, cmap='gray'); axes[1].set_title('Máscara de piezas'); axes[1].axis('off')
plt.show()

## 5. Encontrar contornos y clasificar cada objetivo

`cv2.findContours` busca las fronteras de las regiones blancas de la máscara. Cada contorno puede corresponder a una pieza. El filtro `MIN_CONTOUR_AREA` descarta manchas muy pequeñas.

Para cada contorno usamos `cv2.boundingRect`, que devuelve la caja delimitadora. Después calculamos el centro de esa caja y consultamos el píxel central en HSV. Si su tono indica rojo, la pieza se marca como defectuosa; si indica verde, como conforme. La función retorna tanto la imagen anotada como las predicciones tabulares.

In [ ]:
def detect_and_classify(image_bgr, image_id):
    mask = build_piece_mask(image_bgr)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    annotated = image_bgr.copy()
    predictions = []

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < MIN_CONTOUR_AREA:
            continue
        x, y, width, height = cv2.boundingRect(contour)
        center_x, center_y = x + width // 2, y + height // 2
        hue = int(hsv[center_y, center_x, 0])
        predicted_class = 'defectuosa' if hue <= 12 or hue >= 165 else 'conforme'
        color = (0, 0, 255) if predicted_class == 'defectuosa' else (0, 180, 0)
        cv2.rectangle(annotated, (x, y), (x + width, y + height), color, 2)
        cv2.putText(annotated, predicted_class, (x, max(18, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        predictions.append({'image_id': image_id, 'predicted_class': predicted_class, 'x': x, 'y': y, 'width': width, 'height': height, 'area': area})
    return annotated, pd.DataFrame(predictions)

annotated_image, example_predictions = detect_and_classify(example_image, 0)
print(f'Objetivos detectados en la imagen 0: {len(example_predictions)}')
display(example_predictions)
plt.figure(figsize=(12, 5)); plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)); plt.title('Detección y clasificación'); plt.axis('off'); plt.show()

## 6. Ejecutar el proceso sobre todo el dataset

Ahora aplicamos exactamente la misma función a las 40 imágenes. En una aplicación real, este sería el ciclo que recibe cuadros de una cámara. Guardamos cada predicción en una tabla para poder medir el desempeño y revisar casos concretos.

In [ ]:
prediction_tables = []
annotated_examples = []
for image_id in range(DATASET_SIZE):
    image_path = dataset_dir / f'pieza_{image_id:03d}.png'
    image_bgr = cv2.imread(str(image_path))
    annotated, predictions = detect_and_classify(image_bgr, image_id)
    prediction_tables.append(predictions)
    if image_id < 6:
        annotated_examples.append(annotated)
predictions = pd.concat(prediction_tables, ignore_index=True)
print(f'Predicciones generadas: {len(predictions)}')
display(predictions.head())

## 7. Medir detección y clasificación

Como el dataset fue generado por nosotros, conocemos la respuesta correcta. Para comparar piezas entre verdad y predicción usamos el centro de cada caja; si la distancia entre centros es menor a 25 píxeles, consideramos que es la misma pieza.

Las métricas se interpretan así:

- **Cobertura de detección (recall):** qué proporción de las piezas reales fue encontrada.
- **Precisión de detección:** qué proporción de las detecciones corresponde a piezas reales.
- **Exactitud de clasificación:** qué proporción de las piezas emparejadas recibió la clase correcta.

Estas métricas no sustituyen una validación industrial: aquí no se evalúan desenfoque, piezas parcialmente ocultas ni iluminación extrema.

In [ ]:
matched_rows = []
used_prediction_indices = set()
for _, truth in ground_truth.iterrows():
    truth_center = np.array([truth.x + truth.width / 2, truth.y + truth.height / 2])
    candidates = predictions[predictions.image_id == truth.image_id].copy()
    if candidates.empty:
        continue
    candidates['distance'] = np.sqrt((candidates.x + candidates.width / 2 - truth_center[0]) ** 2 + (candidates.y + candidates.height / 2 - truth_center[1]) ** 2)
    best_index = candidates['distance'].idxmin()
    if candidates.loc[best_index, 'distance'] <= 25 and best_index not in used_prediction_indices:
        used_prediction_indices.add(best_index)
        matched_rows.append({'true_class': truth.class_name, 'predicted_class': predictions.loc[best_index, 'predicted_class'], 'distance': candidates.loc[best_index, 'distance']})

matched = pd.DataFrame(matched_rows)
true_objects = len(ground_truth)
detected_objects = len(predictions)
matched_objects = len(matched)
detection_recall = matched_objects / true_objects
detection_precision = matched_objects / detected_objects
classification_accuracy = (matched.true_class == matched.predicted_class).mean()
print(f'Piezas reales: {true_objects}')
print(f'Piezas detectadas: {detected_objects}')
print(f'Detecciones emparejadas: {matched_objects}')
print(f'Cobertura de detección: {detection_recall:.1%}')
print(f'Precisión de detección: {detection_precision:.1%}')
print(f'Exactitud de clasificación: {classification_accuracy:.1%}')
display(pd.crosstab(matched.true_class, matched.predicted_class, margins=True))

## 8. Interpretar los resultados en lenguaje sencillo

La tabla anterior es una matriz de confusión. Las filas representan la realidad y las columnas la decisión del sistema. Por ejemplo, la celda `conforme / defectuosa` indica piezas buenas que el algoritmo marcó como defectuosas: son falsas alarmas. La celda `defectuosa / conforme` es más delicada, porque representa un defecto que pasó como bueno.

### Cómo leer cada resultado

- **Piezas reales:** es el número de objetos que realmente existen según las etiquetas originales del dataset. Es nuestro punto de comparación.
- **Piezas detectadas:** es el número de objetos que OpenCV encontró después de segmentar la imagen y buscar contornos. Si este número es menor que el real, algunas piezas se perdieron; si es mayor, probablemente aparecieron detecciones de ruido o una pieza fue fragmentada en varias regiones.
- **Detecciones emparejadas:** son las piezas detectadas que pudieron asociarse con una pieza real. El emparejamiento usa la cercanía entre centros para evitar comparar una detección con el objeto equivocado.
- **Cobertura de detección (recall):** responde: `de todas las piezas que sí estaban, ¿cuántas encontré?`. Una cobertura de 100% significa que ninguna pieza real quedó sin localizar. En una planta, una cobertura baja puede significar que el sistema está dejando pasar piezas sin inspección.
- **Precisión de detección:** responde: `de todo lo que marqué como pieza, ¿cuánto era realmente una pieza?`. Una precisión baja significa que el sistema genera falsas detecciones y podría detener la línea innecesariamente o enviar demasiadas piezas a revisión manual.
- **Exactitud de clasificación:** se calcula solamente sobre las piezas que fueron localizadas correctamente. Responde: `de las piezas encontradas, ¿cuántas recibieron la etiqueta correcta?`. Esta métrica no evalúa si se detectó la pieza, sino si se decidió correctamente entre conforme y defectuosa.

### Interpretación operacional para manufactura

Supongamos que el notebook muestra una cobertura de detección de 98% y una exactitud de clasificación de 100%. La lectura correcta sería: casi todas las piezas fueron encontradas y, cuando una pieza fue encontrada, la clase fue asignada correctamente. Sin embargo, el 2% que no se detectó sigue siendo importante: esas piezas no llegaron a la etapa de clasificación y, por lo tanto, no podemos afirmar que fueron inspeccionadas.

Ahora supongamos que la cobertura es 100%, pero la precisión de detección es 85%. En ese caso el sistema encuentra las piezas reales, pero también marca muchos elementos que no son piezas. El sistema sería sensible, pero poco selectivo. En la práctica, produciría alarmas o cajas adicionales y requeriría revisar por qué el ruido del fondo está entrando en la máscara.

También puede ocurrir una exactitud de clasificación de 90%. Esto significa que una de cada diez piezas detectadas recibió una clase incorrecta. Hay dos errores posibles:

- Un **falso rechazo**: una pieza conforme se marca como defectuosa. Aumenta el desperdicio, los retrabajos y las revisiones manuales.
- Un **falso pase**: una pieza defectuosa se marca como conforme. Es normalmente el error más crítico, porque el defecto puede llegar al cliente o a la siguiente operación.

Por eso no basta con reportar un único porcentaje. Conviene revisar por separado cuántas piezas defectuosas fueron clasificadas como conformes. En una aplicación industrial, esa cifra puede ser más importante que la exactitud global.

### Qué significa la matriz de confusión

La matriz permite localizar el tipo exacto de equivocación:

- `conforme / conforme`: decisión correcta para una pieza buena.
- `conforme / defectuosa`: falsa alarma o falso rechazo. La pieza buena fue separada aunque no debía serlo.
- `defectuosa / conforme`: falso pase. El sistema no identificó una pieza que debía rechazarse.
- `defectuosa / defectuosa`: rechazo correcto. El sistema encontró y clasificó adecuadamente el defecto.

Si la matriz tiene valores altos en la diagonal, el sistema está acertando con frecuencia. Si hay valores altos fuera de la diagonal, debemos investigar la causa. Por ejemplo, un reflejo blanco podría reducir el color rojo o verde en el centro de una pieza; un cambio de iluminación podría mover los valores HSV; y dos piezas pegadas podrían producir un único contorno.

### Por qué este resultado no debe generalizarse automáticamente

En este dataset esperamos resultados cercanos al 100% porque el color de las piezas fue diseñado para ser claramente separable. Esto no significa que una solución industrial tendría 100% de desempeño. Significa que la lógica funciona bajo los supuestos controlados del ejercicio. Si cambiamos la iluminación, agregamos reflejos, permitimos piezas tocándose o usamos defectos que no cambian el color, el desempeño puede disminuir.

Además, las imágenes fueron generadas con el mismo procedimiento que luego usamos para evaluarlas. En consecuencia, la prueba es una demostración del funcionamiento del algoritmo, no una certificación de desempeño en producción. Para una evaluación más confiable se necesita separar imágenes de desarrollo y prueba, idealmente tomadas en momentos distintos y con condiciones reales. También es recomendable repetir la medición por lote, turno y condición de iluminación, porque un promedio global puede ocultar fallas en un caso específico.

La conclusión práctica es que el notebook demuestra el flujo completo de una inspección: imagen → máscara → contorno → caja → clase → métrica. Antes de usarlo para tomar decisiones automáticas sobre producto real, habría que calibrar los rangos HSV, validar la cámara, definir una política para casos dudosos y registrar las imágenes rechazadas para auditoría.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for axis, annotated in zip(axes.ravel(), annotated_examples):
    axis.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axis.axis('off')
plt.suptitle('Ejemplos de resultados: rectángulo verde = conforme, rojo = defectuosa', fontsize=14)
plt.tight_layout(); plt.show()

class_counts = ground_truth.class_name.value_counts()
class_counts.plot(kind='bar', color=['#2ca02c', '#d62728'], figsize=(7, 4), title='Distribución real de clases')
plt.ylabel('Número de piezas'); plt.xlabel('Clase'); plt.xticks(rotation=0); plt.show()

## Conclusiones generales

1. OpenCV permite construir una inspección visual básica sin entrenar una red neuronal. En este caso, segmentación por color y contornos son suficientes porque el entorno está controlado.

2. El sistema realiza dos tareas distintas: **detección**, que responde dónde está la pieza, y **clasificación**, que responde qué tipo de pieza es. Una métrica alta en una tarea no garantiza automáticamente una métrica alta en la otra.

3. El dataset sintético es útil para aprender y probar el flujo completo, pero no representa toda la variabilidad de una planta. Para pasar a producción sería necesario recopilar imágenes reales con diferentes turnos, lotes, cámaras, posiciones, suciedad, reflejos y niveles de iluminación.

4. El umbral `MIN_CONTOUR_AREA` y los rangos HSV son parámetros de proceso. Deben calibrarse con datos reales. Un umbral demasiado bajo genera falsas detecciones por ruido; uno demasiado alto puede ignorar piezas pequeñas.

5. Un siguiente nivel sería detectar defectos geométricos o superficiales que no puedan separarse por color. Para eso se podrían usar características de forma, comparación contra una plantilla o modelos de aprendizaje profundo como YOLO.

6. Antes de usar el sistema para liberar producto, conviene definir el costo de cada error. En manufactura, dejar pasar una pieza defectuosa normalmente es más grave que detener una pieza buena para revisión manual. Por ello, el umbral y la lógica deben priorizar el riesgo del proceso.

En resumen: este notebook muestra de manera reproducible cómo una imagen puede transformarse en una decisión de inspección. También deja claro que el éxito técnico depende tanto del algoritmo como de la calidad de las imágenes y de la definición operacional de lo que significa 'defectuoso'.

## Cómo ejecutar en Google Colab

1. Abre [Google Colab](https://colab.research.google.com/).
2. Selecciona **Archivo → Subir notebook** y elige este archivo `.ipynb`.
3. Ejecuta las celdas en orden con `Shift + Enter`.
4. Observa primero las imágenes del dataset, después la máscara, las cajas detectadas y finalmente las métricas.

El notebook no requiere GPU ni archivos adicionales.